# MINI PROJECT: CUSTOMER CHURN ANALYSIS

**Objective:** Analyze customer demographics, services, contracts, billing and churn behavior and identify patterns associated with customer churn.

**Dataset:** A synthetic telecom/subscription customer dataset created for this academic mini-project with ChatGPT assistance. It follows the suggested assignment schema and contains 7,043 customer records.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
ROOT = Path.cwd().parent if Path.cwd().name == "Python" else Path.cwd()
DATASET = ROOT / "Dataset" / "Customer_Churn_Cleaned.csv"
VIZ = ROOT / "Visualizations"
VIZ.mkdir(exist_ok=True)
df = pd.read_csv(DATASET)


## PART 1 — DATA LOADING

In [ ]:
# 1–5. First five, last five, shape and column names
print("First five records:")
display(df.head())
print("Last five records:")
display(df.tail())
print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())


## PART 2 — DATA EXPLORATION & CLEANING

In [ ]:
# 7–16. Data types, statistics, missing values, duplicates, unique values, cleaning and tenure groups
print("Data types:")
display(df.dtypes)
print("Statistical information:")
display(df.describe(include="all").T)
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False))
print("Duplicate records:", df.duplicated().sum())

# Remove duplicates if any
df = df.drop_duplicates().copy()

# Convert appropriate columns
for c in ["Senior_Citizen", "Tenure_Months"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in ["Monthly_Charges", "Total_Charges"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Handle missing numeric values
df["Total_Charges"] = df["Total_Charges"].fillna(df["Monthly_Charges"] * df["Tenure_Months"])

# Create tenure groups
df["Tenure_Group"] = pd.cut(
    df["Tenure_Months"], bins=[-1,12,24,48,72],
    labels=["0–12 Months","13–24 Months","25–48 Months","49–72 Months"]
)

print("Unique values by selected column:")
for c in ["Gender","Contract","Internet_Service","Payment_Method","Churn"]:
    print(c, df[c].unique().tolist())

print("Cleaned shape:", df.shape)
print("Remaining missing values:", int(df.isna().sum().sum()))
df.to_csv(DATASET,index=False)


## PART 3 — DATA ANALYSIS

In [ ]:
def churn_rate_by(col):
    return (df.groupby(col, observed=True)["Churn"]
              .apply(lambda s: (s == "Yes").mean() * 100)
              .round(2).sort_values(ascending=False))

# 17–23. Core KPIs
total_customers = len(df)
total_churned = int((df["Churn"] == "Yes").sum())
total_retained = int((df["Churn"] == "No").sum())
overall_churn_rate = total_churned / total_customers * 100
avg_monthly = df["Monthly_Charges"].mean()
avg_total = df["Total_Charges"].mean()
avg_tenure = df["Tenure_Months"].mean()

print("17. Total customers:", total_customers)
print("18. Total churned customers:", total_churned)
print("19. Total retained customers:", total_retained)
print(f"20. Overall churn rate: {overall_churn_rate:.2f}%")
print(f"21. Average monthly charges: {avg_monthly:.2f}")
print(f"22. Average total charges: {avg_total:.2f}")
print(f"23. Average customer tenure: {avg_tenure:.2f} months")

print("24. Customers by gender:"); display(df["Gender"].value_counts())
print("25. Customers by contract type:"); display(df["Contract"].value_counts())
print("26. Customers by internet service:"); display(df["Internet_Service"].value_counts())
print("27. Customers by payment method:"); display(df["Payment_Method"].value_counts())
print("28. Churn rate by gender:"); display(churn_rate_by("Gender"))
print("29. Churn rate by contract type:"); display(churn_rate_by("Contract"))
print("30. Churn rate by internet service:"); display(churn_rate_by("Internet_Service"))
print("31. Churn rate by payment method:"); display(churn_rate_by("Payment_Method"))
print("32. Churn rate by senior citizen status:"); display(churn_rate_by("Senior_Citizen"))
print("33. Churn rate by tenure group:"); display(churn_rate_by("Tenure_Group"))

print("34. Monthly charges — churned vs retained:")
display(df.groupby("Churn")["Monthly_Charges"].agg(["mean","median"]).round(2))
print("35. Tenure — churned vs retained:")
display(df.groupby("Churn")["Tenure_Months"].agg(["mean","median"]).round(2))
print("36. Top 10 customers by total charges:")
display(df.nlargest(10,"Total_Charges")[["Customer_ID","Tenure_Months","Monthly_Charges","Total_Charges","Churn"]])

segment_tables = {
    "Contract": churn_rate_by("Contract"),
    "Internet_Service": churn_rate_by("Internet_Service"),
    "Payment_Method": churn_rate_by("Payment_Method"),
    "Tenure_Group": churn_rate_by("Tenure_Group"),
    "Gender": churn_rate_by("Gender"),
    "Senior_Citizen": churn_rate_by("Senior_Citizen")
}
print("37. Highest-churn customer segments:")
for k,v in segment_tables.items(): print(k, "->", v.index[0], f"({v.iloc[0]:.2f}%)")
print("38. Major factors associated with churn: contract type, internet service, payment method, tenure, senior-citizen status and monthly charges are examined below.")


## PART 4 — PYTHON VISUALIZATION (14 REQUIRED)

In [ ]:
def savefig(name, title):
    plt.title(title, fontweight="bold")
    plt.tight_layout()
    plt.savefig(VIZ / name, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close()

# 1. Churn Distribution — Pie Chart
df["Churn"].value_counts().plot(kind="pie", autopct="%1.1f%%", startangle=90); plt.ylabel(""); savefig("01_churn_distribution.png","Customer Churn Distribution")
# 2. Customer Distribution by Contract — Bar Chart
df["Contract"].value_counts().plot(kind="bar"); plt.xlabel("Contract"); plt.ylabel("Customers"); savefig("02_customer_by_contract.png","Customer Distribution by Contract")
# 3. Churn by Contract — Bar Chart
churn_rate_by("Contract").sort_values().plot(kind="bar"); plt.xlabel("Contract"); plt.ylabel("Churn Rate (%)"); savefig("03_churn_by_contract.png","Churn Rate by Contract")
# 4. Churn by Gender — Bar Chart
churn_rate_by("Gender").sort_values().plot(kind="bar"); plt.xlabel("Gender"); plt.ylabel("Churn Rate (%)"); savefig("04_churn_by_gender.png","Churn Rate by Gender")
# 5. Churn by Internet Service — Bar Chart
churn_rate_by("Internet_Service").sort_values().plot(kind="bar"); plt.xlabel("Internet Service"); plt.ylabel("Churn Rate (%)"); savefig("05_churn_by_internet.png","Churn Rate by Internet Service")
# 6. Churn by Payment Method — Bar Chart
churn_rate_by("Payment_Method").sort_values().plot(kind="bar"); plt.xlabel("Payment Method"); plt.ylabel("Churn Rate (%)"); plt.xticks(rotation=25,ha="right"); savefig("06_churn_by_payment.png","Churn Rate by Payment Method")
# 7. Tenure Distribution — Histogram
plt.hist(df["Tenure_Months"], bins=15); plt.xlabel("Tenure (Months)"); plt.ylabel("Customers"); savefig("07_tenure_histogram.png","Tenure Distribution")
# 8. Monthly Charges Distribution — Histogram
plt.hist(df["Monthly_Charges"], bins=20); plt.xlabel("Monthly Charges"); plt.ylabel("Customers"); savefig("08_monthly_charges_histogram.png","Monthly Charges Distribution")
# 9. Tenure vs Monthly Charges — Scatter Plot
plt.scatter(df["Tenure_Months"], df["Monthly_Charges"], alpha=.18, s=10); plt.xlabel("Tenure (Months)"); plt.ylabel("Monthly Charges"); savefig("09_tenure_vs_monthly.png","Tenure vs Monthly Charges")
# 10. Monthly Charges by Churn — Box Plot
plt.boxplot([df.loc[df.Churn=="No","Monthly_Charges"],df.loc[df.Churn=="Yes","Monthly_Charges"]], labels=["Retained","Churned"]); plt.ylabel("Monthly Charges"); savefig("10_monthly_charges_by_churn.png","Monthly Charges by Churn")
# 11. Total Charges by Churn — Box Plot
plt.boxplot([df.loc[df.Churn=="No","Total_Charges"],df.loc[df.Churn=="Yes","Total_Charges"]], labels=["Retained","Churned"]); plt.ylabel("Total Charges"); savefig("11_total_charges_by_churn.png","Total Charges by Churn")
# 12. Churn Rate by Tenure Group — Bar Chart
churn_rate_by("Tenure_Group").reindex(["0–12 Months","13–24 Months","25–48 Months","49–72 Months"]).plot(kind="bar"); plt.xlabel("Tenure Group"); plt.ylabel("Churn Rate (%)"); savefig("12_churn_by_tenure_group.png","Churn Rate by Tenure Group")
# 13. Service Usage — Count Plot
services=["Phone_Service","Online_Security","Online_Backup","Device_Protection","Tech_Support","Streaming_TV","Streaming_Movies"]
service_yes=[int((df[c]=="Yes").sum()) for c in services]
plt.bar([s.replace("_"," ") for s in services],service_yes); plt.xticks(rotation=30,ha="right"); plt.ylabel("Customers Using Service"); savefig("13_service_usage.png","Service Usage")
# 14. Correlation Heatmap
num=df[["Senior_Citizen","Tenure_Months","Monthly_Charges","Total_Charges"]].copy(); num["Churn"]=(df.Churn=="Yes").astype(int); num["Partner"]=(df.Partner=="Yes").astype(int); num["Dependents"]=(df.Dependents=="Yes").astype(int)
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0); savefig("14_correlation_heatmap.png","Correlation Heatmap")


## PART 5 — POWER BI DASHBOARD

Dashboard title: **Customer Churn & Retention Analytics Dashboard**

Required KPI cards: Total Customers, Churned Customers, Retained Customers, Churn Rate %, Average Monthly Charges, Average Tenure.

Required visuals: Churn Distribution (Donut), Churn Rate by Contract, Churn Rate by Internet Service, Churn Rate by Payment Method, Customer Count by Tenure Group, Monthly Charges by Churn, Customer Distribution by Contract, Customer Distribution by Gender, Tenure vs Monthly Charges, Customer Details table/matrix.

Required slicers: Gender, Contract, Internet Service, Payment Method, Senior Citizen, Churn.


## PART 9 — DAX REQUIREMENT

Use these measures after importing `Customer_Churn_Cleaned.csv` into Power BI. Replace the table name if Power BI assigns a different name.

```DAX
Total Customers = COUNTROWS(Customer_Churn_Cleaned)
Churned Customers = CALCULATE([Total Customers], Customer_Churn_Cleaned[Churn] = "Yes")
Retained Customers = CALCULATE([Total Customers], Customer_Churn_Cleaned[Churn] = "No")
Churn Rate % = DIVIDE([Churned Customers], [Total Customers], 0)
Average Monthly Charges = AVERAGE(Customer_Churn_Cleaned[Monthly_Charges])
Average Tenure = AVERAGE(Customer_Churn_Cleaned[Tenure_Months])
```

## PART 10 — BUSINESS INSIGHTS

In [ ]:
print(f"1. Overall churn rate: {overall_churn_rate:.2f}%")
print("2. Contract type with highest churn:", churn_rate_by("Contract").index[0])
print("3. Payment method with highest churn:", churn_rate_by("Payment_Method").index[0])
print("4. Internet service with highest churn:", churn_rate_by("Internet_Service").index[0])
print("5. Tenure relationship: compare the tenure-group churn rates; shorter tenure is treated as higher risk when its rate is highest.")
print("6. Monthly-charge relationship: compare mean/median monthly charges for churned and retained customers.")
print("7. Highest-risk customer segment:", churn_rate_by("Tenure_Group").index[0], "tenure group")
print("8. Major factors examined: contract, internet service, payment method, tenure, senior-citizen status and charges.")
print("9. Segment needing attention: the highest-churn contract/internet/payment/tenure combinations.")
print("10. Use the tables and 14 charts above to write 5–10 key business insights in the report.")


## CONCLUSION

The project completes the requested end-to-end workflow: data loading, exploration, cleaning, analysis, 14 Python visualizations, Power BI dashboard design, DAX measures and business insights. The cleaned CSV in the Dataset folder is the source for Power BI.
